# Dependent Events and Probability Trees

This notebook applies the general product rule $P(A \cap B) = P(A) \cdot P(B \mid A)$ to **dependent events**, and introduces the **probability tree** — a diagram that organizes all scenarios and their probabilities. It is based on the ninth video of Week 1 (Introduction to Probability), a continuation of the conditional-probability lesson.


## 1. Introduction

In notebook 06, kids were split into rooms **randomly**, which made "plays soccer" and "is in room 1" independent. But suppose room 1 has a TV showing the **World Cup** and room 2 shows an unrelated movie — and kids choose their room. Now we'd expect the soccer fans to flock to room 1: the events have become **dependent**, and the probabilities change.

After completing this notebook you will be able to:

- Tell dependent from independent events in everyday situations.
- Compute joint probabilities of dependent events with the general product rule.
- Build a probability tree and read off the probability of every scenario.
- Check that the leaves of a probability tree sum to 1.


## 2. Theory

### Dependence

Events are **dependent** when the occurrence of one changes the probability of the other. Choosing the World-Cup room is much more likely *if* you like soccer, so the events are dependent. A random room assignment carries no information about soccer preference, so those events are independent.

### The general product rule, again

For any events,

$$
P(A \cap B) = P(A) \cdot P(B \mid A)
$$

For dependent events the conditional factor $P(B \mid A)$ really matters — replacing it with $P(B)$ (the independent shortcut) gives the wrong answer.

### Probability trees

A **probability tree** organizes a two-stage experiment:

- The first level branches on the first event (soccer / not soccer), labeled with its probabilities.
- Each branch then splits on the second event (running shoes / not), labeled with the **conditional** probabilities given that branch.
- The probability of each **leaf** (complete scenario) is the product of the probabilities along its path.

Because the leaves cover every possible scenario exactly once, their probabilities **sum to 1**.


## 3. Worked Examples

Setup for the examples: a school with 100 kids, **40 play soccer** ($P(S) = 0.4$, so $P(S') = 0.6$).

### Example 1: Soccer and running shoes

Among the kids who play soccer, **80% wear running shoes**: $P(R \mid S) = 0.8$.

How many kids play soccer *and* wear running shoes? 80% of 40 is 32:

$$
P(S \cap R) = P(S) \cdot P(R \mid S) = 0.4 \times 0.8 = 0.32
$$

### Example 2: No soccer, but running shoes

Among the kids who **don't** play soccer, 50% wear running shoes: $P(R \mid S') = 0.5$.

$$
P(S' \cap R) = P(S') \cdot P(R \mid S') = 0.6 \times 0.5 = 0.30
$$

### Example 3: The full probability tree

| Scenario | Product | Probability |
|---|---|---|
| Soccer, running shoes | $0.4 \times 0.8$ | 0.32 |
| Soccer, no running shoes | $0.4 \times 0.2$ | 0.08 |
| No soccer, running shoes | $0.6 \times 0.5$ | 0.30 |
| No soccer, no running shoes | $0.6 \times 0.5$ | 0.30 |

The four leaves sum to $0.32 + 0.08 + 0.30 + 0.30 = 1$. ✓

Note the dependence: wearing running shoes is more likely for soccer players (80%) than for non-players (50%) — knowing one event shifts the probability of the other.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)


### The tree as a computation


In [ ]:
p_s = 0.4          # P(soccer)
p_r_given_s = 0.8  # P(running shoes | soccer)
p_r_given_ns = 0.5 # P(running shoes | not soccer)

leaves = {
    "soccer ∩ shoes":        p_s * p_r_given_s,
    "soccer ∩ no shoes":     p_s * (1 - p_r_given_s),
    "no soccer ∩ shoes":     (1 - p_s) * p_r_given_ns,
    "no soccer ∩ no shoes":  (1 - p_s) * (1 - p_r_given_ns),
}

for name, p in leaves.items():
    print(f"P({name}) = {p:.2f}")
print(f"Sum of all leaves: {sum(leaves.values()):.2f}")


### Simulating dependent events

To simulate dependence we draw the second event's outcome with a probability that **depends on** the first event's outcome.


In [ ]:
n_kids = 1_000_000

plays_soccer = rng.random(n_kids) < p_s
p_shoes = np.where(plays_soccer, p_r_given_s, p_r_given_ns)
wears_shoes = rng.random(n_kids) < p_shoes

print(f"Simulated P(S ∩ R)  = {(plays_soccer & wears_shoes).mean():.4f}   (theory 0.32)")
print(f"Simulated P(S' ∩ R) = {(~plays_soccer & wears_shoes).mean():.4f}   (theory 0.30)")

# Dependence check: P(R | S) vs P(R | S') in the simulation
p_r_s = wears_shoes[plays_soccer].mean()
p_r_ns = wears_shoes[~plays_soccer].mean()
print(f"Simulated P(R | S)  = {p_r_s:.4f} vs P(R | S') = {p_r_ns:.4f} — "
      "different, so the events are dependent")


## 5. Visualizations

### The probability tree


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

root = (0, 3)
level1 = {"S": (3, 4.5), "S'": (3, 1.5)}
level2 = {
    "S ∩ R": (7, 5.4), "S ∩ R'": (7, 3.6),
    "S' ∩ R": (7, 2.4), "S' ∩ R'": (7, 0.6),
}

def branch(a, b, label):
    ax.plot([a[0], b[0]], [a[1], b[1]], color="gray", linewidth=1.5)
    mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
    ax.annotate(label, (mx, my), textcoords="offset points", xytext=(0, 7),
                ha="center", fontsize=10, color="darkblue")

branch(root, level1["S"], "P(S) = 0.4")
branch(root, level1["S'"], "P(S') = 0.6")
branch(level1["S"], level2["S ∩ R"], "P(R|S) = 0.8")
branch(level1["S"], level2["S ∩ R'"], "P(R'|S) = 0.2")
branch(level1["S'"], level2["S' ∩ R"], "P(R|S') = 0.5")
branch(level1["S'"], level2["S' ∩ R'"], "P(R'|S') = 0.5")

ax.plot(*root, "o", markersize=10, color="black")
ax.annotate("100 kids", root, textcoords="offset points", xytext=(-10, 0),
            ha="right", fontsize=11)

for name, pos in level1.items():
    ax.plot(*pos, "o", markersize=9, color="#4e79a7")
    ax.annotate("soccer" if name == "S" else "no soccer", pos,
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=10)

leaf_probs = {"S ∩ R": 0.32, "S ∩ R'": 0.08, "S' ∩ R": 0.30, "S' ∩ R'": 0.30}
for name, pos in level2.items():
    ax.plot(*pos, "s", markersize=9, color="#e15759")
    ax.annotate(f"{name}: {leaf_probs[name]:.2f}", pos,
                textcoords="offset points", xytext=(12, 0), ha="left", fontsize=11)

ax.set_xlim(-1.5, 9.5)
ax.set_ylim(0, 6.2)
ax.axis("off")
ax.set_title("Probability tree: multiply along each path; the four leaves sum to 1")
plt.show()


### Independent vs. dependent, as a picture

Divide a square by soccer preference (vertical split) and by room/shoes (horizontal split within each group). If the split lines **line up across groups**, the events are independent — the fraction is the same in both groups. If they **don't line up**, the events are dependent.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: random rooms — same 50% split for both groups (independent)
# Right: running shoes — 80% of soccer kids vs 50% of the rest (dependent)
cases = [
    ("Random rooms: independent", 0.5, 0.5, 0.5, ["room 1", "room 2"]),
    ("Running shoes: dependent", 0.4, 0.8, 0.5, ["shoes", "no shoes"]),
]

for ax, (title, split_x, frac_left, frac_right, labels) in zip(axes, cases):
    ax.add_patch(plt.Rectangle((0, 0), split_x, frac_left,
                               facecolor="#7fc97f", edgecolor="white"))
    ax.add_patch(plt.Rectangle((0, frac_left), split_x, 1 - frac_left,
                               facecolor="#c7e9c0", edgecolor="white"))
    ax.add_patch(plt.Rectangle((split_x, 0), 1 - split_x, frac_right,
                               facecolor="#fdc086", edgecolor="white"))
    ax.add_patch(plt.Rectangle((split_x, frac_right), 1 - split_x, 1 - frac_right,
                               facecolor="#fee8c8", edgecolor="white"))

    ax.axvline(split_x, color="black", linewidth=1.5)
    ax.plot([0, split_x], [frac_left, frac_left], color="black", linewidth=2)
    ax.plot([split_x, 1], [frac_right, frac_right], color="black", linewidth=2)

    ax.text(split_x / 2, -0.06, "soccer", ha="center", fontsize=10)
    ax.text((1 + split_x) / 2, -0.06, "no soccer", ha="center", fontsize=10)
    ax.text(split_x / 2, frac_left / 2, labels[0], ha="center", fontsize=10)
    ax.text(split_x / 2, (1 + frac_left) / 2, labels[1], ha="center", fontsize=10)

    ax.set_xlim(0, 1)
    ax.set_ylim(-0.12, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

plt.suptitle("Independent events: the horizontal split lines meet. Dependent: they don't.")
plt.tight_layout()
plt.show()


## 6. Exercises

### Basic

1. Using the tree above, what is the overall probability that a random kid wears running shoes? (Add the leaves where $R$ occurs.)
2. Are "plays soccer" and "wears running shoes" independent here? Justify with numbers.

### Intermediate

3. A drawer has 60% blue socks and 40% black socks. If it rains, a kid wears boots with probability 0.9; if it doesn't rain, with probability 0.2. The probability of rain is 0.3. Draw the tree (mentally or on paper) and compute the probability of each of the four scenarios. Do they sum to 1?
4. In the school example, what is $P(S \mid R)$ — the probability that a kid plays soccer *given* that they wear running shoes? (Use the leaf probabilities; this is a first taste of Bayes' theorem.)

### Advanced

5. Simulate the World-Cup-room scenario: 100 kids, 50 soccer fans; a soccer fan goes to room 1 with probability 0.9, a non-fan with probability 0.1. Estimate $P(\text{soccer} \mid \text{room 1})$ and compare it with $P(\text{soccer}) = 0.5$.


## 7. Solutions


In [ ]:
# Exercise 1: total probability of running shoes
p_r = leaves["soccer ∩ shoes"] + leaves["no soccer ∩ shoes"]
print("1.  P(R) = 0.32 + 0.30 =", round(p_r, 2))

# Exercise 2: compare P(R | S) with P(R)
print("2.  P(R | S) = 0.8 ≠ P(R) = 0.62 — dependent events.")

# Exercise 3: rain and boots
p_rain = 0.3
scenarios = {
    "rain ∩ boots":       p_rain * 0.9,
    "rain ∩ no boots":    p_rain * 0.1,
    "no rain ∩ boots":    (1 - p_rain) * 0.2,
    "no rain ∩ no boots": (1 - p_rain) * 0.8,
}
for name, p in scenarios.items():
    print(f"3.  P({name}) = {p:.2f}")
print("    sum =", round(sum(scenarios.values()), 2))

# Exercise 4: P(S | R) = P(S ∩ R) / P(R)
print("4.  P(S | R) = 0.32 / 0.62 =", round(0.32 / 0.62, 4))

# Exercise 5: the World-Cup room
n = 1_000_000
fan = rng.random(n) < 0.5
p_room1 = np.where(fan, 0.9, 0.1)
in_room1 = rng.random(n) < p_room1
p_fan_given_room1 = fan[in_room1].mean()
print(f"5.  P(soccer | room 1) ≈ {p_fan_given_room1:.4f} vs P(soccer) = 0.5 — "
      "knowing the room strongly changes the probability.")


## 8. Conclusion

- Events are **dependent** when one changes the probability of the other — like choosing the World-Cup room when you love soccer, or wearing running shoes when you play.
- The general product rule handles dependence: $P(A \cap B) = P(A) \cdot P(B \mid A)$.
- A **probability tree** turns a two-stage problem into paths: multiply along each path to get a scenario's probability, and all leaves sum to 1.
- Reversing a tree — asking $P(S \mid R)$ when the tree gives $P(R \mid S)$ — is exactly what **Bayes' theorem** does, coming up next.
